In [ ]:
# BarçaIQ — Notebook 7 · Module 5: LLM RAG Tactical Assistant
## AI-powered tactical Q&A grounded in real StatsBomb data

**What this builds:** A Retrieval-Augmented Generation system that lets
coaching staff ask natural language tactical questions and get answers
grounded in real Barça data — not hallucinations.

**How RAG works:**
1. All findings + match data → chunked → embedded → stored in FAISS vector DB
2. Coach asks a question → question embedded → most relevant chunks retrieved
3. Retrieved context + question → sent to Gemini → grounded tactical answer

| | |
|---|---|
| **Knowledge base** | Possession patterns · Press findings · Era comparisons · GNN predictions |
| **LLM** | Gemini 1.5 Flash (free tier) |
| **Vector store** | FAISS (local, no server needed) |
| **Saves to** | `BarçaIQ/rag/` |
| **Runtime** | CPU is fine |

---

In [ ]:
!pip install langchain langchain-google-genai langchain-community faiss-cpu sentence-transformers -q

print("✅ Done — restart runtime if this is a fresh session")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
✅ Done — restart runtime if this is a fresh session


In [ ]:
!pip show langchain langchain-community langchain-google-genai | grep -E "Name|Version"

Name: langchain
Version: 1.2.12
Name: langchain-community
Version: 0.4.1
Name: langchain-google-genai
Version: 4.2.1


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, json, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Load API key from Colab secrets
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
os.environ['GOOGLE_API_KEY'] = GEMINI_API_KEY

base    = "/content/drive/MyDrive/BarçaIQ"
rag_dir = f"{base}/rag"
os.makedirs(rag_dir, exist_ok=True)

print("✅ Imports ready")
print(f"✅ API key loaded : {'*' * 20}{GEMINI_API_KEY[-4:]}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Imports ready
✅ API key loaded : ********************GddI


In [ ]:
import pandas as pd

passes    = pd.read_csv('/content/drive/MyDrive/BarçaIQ/data/barca_passes_all_eras.csv')
sequences = pd.read_csv('/content/drive/MyDrive/BarçaIQ/data/possession_sequences_all_eras.csv')

print("Passes columns:", passes.columns.tolist())
print("Sequences columns:", sequences.columns.tolist())
print("\nPasses shape:", passes.shape)
print("\nSample passes:\n", passes.head(2))

Passes columns: ['50_50', 'bad_behaviour_card', 'ball_receipt_outcome', 'ball_recovery_offensive', 'ball_recovery_recovery_failure', 'block_deflection', 'carry_end_location', 'clearance_aerial_won', 'clearance_body_part', 'clearance_head', 'clearance_left_foot', 'clearance_right_foot', 'counterpress', 'dribble_no_touch', 'dribble_nutmeg', 'dribble_outcome', 'dribble_overrun', 'duel_outcome', 'duel_type', 'duration', 'foul_committed_advantage', 'foul_committed_card', 'foul_committed_offensive', 'foul_committed_type', 'foul_won_advantage', 'foul_won_defensive', 'goalkeeper_body_part', 'goalkeeper_end_location', 'goalkeeper_outcome', 'goalkeeper_position', 'goalkeeper_technique', 'goalkeeper_type', 'id', 'index', 'interception_outcome', 'location', 'match_id', 'minute', 'off_camera', 'out', 'pass_aerial_won', 'pass_angle', 'pass_assisted_shot_id', 'pass_body_part', 'pass_cross', 'pass_cut_back', 'pass_deflected', 'pass_end_location', 'pass_goal_assist', 'pass_height', 'pass_inswinging', '

In [ ]:
print(json.dumps(era_dna, indent=2))

{
  "pep": {
    "mean_shot_prob": 0.3845,
    "actual_shot_rate": 0.1485,
    "n_sequences": 9997
  },
  "msn": {
    "mean_shot_prob": 0.3997,
    "actual_shot_rate": 0.1532,
    "n_sequences": 7637
  }
}


In [ ]:
documents = []

# ── 1. Press trigger findings (Module 3) ─────────────────────────────────────
with open(f"{base}/models/press_trigger_findings.json") as f:
    press_findings = json.load(f)

documents.append(Document(
    page_content=f"""
    PRESS TRIGGER ANALYSIS — FC BARCELONA (Pep 2008-12, MSN 2014-17)

    Model performance: AUC {press_findings['model']['test_auc']},
    F1 {press_findings['model']['test_f1']},
    Accuracy {press_findings['model']['test_accuracy']}

    Overall press success rate: {press_findings['key_findings']['overall_press_success_rate']}
    Pep era press success rate: {press_findings['key_findings']['pep_press_success_rate']}
    MSN era press success rate: {press_findings['key_findings']['msn_press_success_rate']}
    Counterpress success rate: {press_findings['key_findings']['counterpress_success_rate']}
    Normal organised press success rate: {press_findings['key_findings']['normal_press_success_rate']}
    Final third press success: {press_findings['key_findings']['final_third_press_success']}
    Mid third press success: {press_findings['key_findings']['mid_third_press_success']}

    Press success by number of teammates pressing simultaneously:
    1 player: {press_findings['teammates_pressing_success']['1']}
    2 players: {press_findings['teammates_pressing_success']['2']}
    3 players: {press_findings['teammates_pressing_success']['3']}
    4-5 players: {press_findings['teammates_pressing_success']['4-5']}
    6+ players: {press_findings['teammates_pressing_success']['6+']}

    Most important features predicting press success:
    1. press_index_in_possession (0.458) — timing of press in opponent possession
    2. teammates_pressing (0.355) — number of players pressing together
    3. is_counterpress (0.074) — whether it is immediate counterpress

    Recommendations for Hansi Flick:
    {chr(10).join(f'- {r}' for r in press_findings['flick_recommendations'])}
    """,
    metadata={"source": "module3_press_analysis", "topic": "pressing"}
))

# ── 2. Era DNA summary (Module 2) ─────────────────────────────────────────────
with open(f"{base}/models/era_dna_summary.json") as f:
    era_dna = json.load(f)

documents.append(Document(
    page_content=f"""
    ERA TACTICAL DNA — GNN ANALYSIS

    GNN Model: GraphSAGE, AUC 0.782, F1 0.422, Accuracy 0.762
    The GNN was trained on 17,634 possession sequences from both eras.

    MSN Era (2014-17) — Luis Enrique:
    GNN predicted shot probability: {era_dna['msn']['mean_shot_prob']}
    Actual shot ending rate: {era_dna['msn']['actual_shot_rate']}
    Total sequences analysed: {era_dna['msn']['n_sequences']}

    Pep Era (2008-12) — Pep Guardiola:
    GNN predicted shot probability: {era_dna['pep']['mean_shot_prob']}
    Actual shot ending rate: {era_dna['pep']['actual_shot_rate']}
    Total sequences analysed: {era_dna['pep']['n_sequences']}

    Finding: MSN era passing graph structure is marginally more shot-threatening
    than Pep era (0.400 vs 0.385 predicted shot probability).
    MSN used longer sequences (9.28 vs 9.00 passes), wider structure
    (6.42 vs 6.26 players per sequence), and had 3.2% better shot conversion.
    Pep had 5.2% more sequences per match — higher volume, shorter combinations.
    """,
    metadata={"source": "module2_gnn", "topic": "tactical_dna"}
))

# ── 3. Tactical pattern findings ──────────────────────────────────────────────
pattern_df    = pd.read_csv(f"{base}/data/possession_patterns.csv")
pattern_stats = pattern_df.groupby('pattern').agg(
    count    =('pattern', 'count'),
    shot_rate=('ended_with_shot', 'mean')
).round(3)

documents.append(Document(
    page_content=f"""
    BARÇA TACTICAL PATTERN ANALYSIS

    5 core Barça patterns identified from {len(pattern_df):,} possession sequences:

    1. THIRD-MAN COMBINATION
       Count: {pattern_stats.loc['third_man_combo', 'count']:,} sequences
       Shot rate: {pattern_stats.loc['third_man_combo', 'shot_rate']:.1%}
       Definition: Player A passes to B who immediately plays C in final third
       Significance: Most frequent AND most dangerous pattern

    2. FALSE NINE DROP
       Count: {pattern_stats.loc['false_nine_drop', 'count']:,} sequences
       Shot rate: {pattern_stats.loc['false_nine_drop', 'shot_rate']:.1%}
       Definition: Forward drops to midfield zone (x=40-75) then plays
                   forward pass gaining 15+ pitch units
       Era split: MSN used this 62% more than Pep (366 vs 226 sequences)

    3. INVERTED WINGER
       Count: {pattern_stats.loc['inverted_winger', 'count']:,} sequences
       Shot rate: {pattern_stats.loc['inverted_winger', 'shot_rate']:.1%}
       Definition: Wide player receives out wide then cuts 15+ units inside
       Era split: Pep era dominant (784 vs 637 sequences)

    4. TIKI-TAKA BUILDUP
       Count: {pattern_stats.loc['tiki_taka_buildup', 'count']:,} sequences
       Shot rate: {pattern_stats.loc['tiki_taka_buildup', 'shot_rate']:.1%}
       Definition: 10+ passes, 6+ players, positive pitch progression

    5. DIRECT ATTACK
       Count: {pattern_stats.loc['direct_attack', 'count']:,} sequences
       Shot rate: {pattern_stats.loc['direct_attack', 'shot_rate']:.1%}
       Definition: 3-5 passes covering 50+ pitch units forward

    6. OTHER (possession keeping)
       Count: {pattern_stats.loc['other', 'count']:,} sequences
       Shot rate: {pattern_stats.loc['other', 'shot_rate']:.1%}
       Significance: Pure possession keeping has lowest shot rate —
       confirms Juego de Posicion is about specific patterns not just
       keeping the ball.

    Key insight for Flick: Third-man combinations (22.7%) and false nine
    drops (21.1%) are nearly 50% more likely to end in a shot than the
    dataset average (15.1%). Coaching should prioritise drilling these
    two patterns specifically.
    """,
    metadata={"source": "module2_patterns", "topic": "tactical_patterns"}
))

# ── 4. Barça philosophy + Flick context ───────────────────────────────────────
documents.append(Document(
    page_content="""
    FC BARCELONA — JUEGO DE POSICIÓN PHILOSOPHY

    Core principles of Barça's playing style:

    1. POSITIONAL SUPERIORITY — Always have a free man through movement
       and spacing before the ball arrives. Create numerical, positional,
       and qualitative superiorities.

    2. PRESSING AS A TEAM — Press is triggered by specific cues, not
       individual decisions. The whole team presses as one unit when
       the trigger occurs (poor touch, back pass to keeper, etc.)

    3. THIRD-MAN PRINCIPLE — Direct passes are often decoys. The real
       danger comes from the third man making a run while the opponent
       tracks the direct pass.

    4. JUEGO DE POSICIÓN — Occupy specific zones of the pitch to create
       passing triangles and diamonds. Always have passing options at
       different angles and distances.

    5. IMMEDIATE PRESS AFTER LOSS — When possession is lost, press
       immediately within 5 seconds to recover the ball before the
       opponent can organise. However data shows this has only 21%
       success — organised positional press is more effective.

    6. FALSE NINE — Forward drops deep to attract centre-backs,
       creating space in behind for midfielders to run into.
       Pioneered by Pep with Messi, continued under Luis Enrique.

    Hansi Flick took over as Barcelona head coach in 2024.
    His system is characterised by aggressive high pressing,
    quick vertical transitions, and a high defensive line.
    This system is more intense than either the Pep (2008-12)
    or MSN (2014-17) eras analysed in this system.

    Note: This RAG system's statistical knowledge is based on
    StatsBomb data from Pep era (2008-12) and MSN era (2014-17).
    Findings are applied as principles to Flick's current system,
    not as direct current-squad analysis.
    """,
    metadata={"source": "barca_philosophy", "topic": "philosophy"}
))

print(f"✅ Knowledge base built : {len(documents)} documents")
for doc in documents:
    print(f"   {doc.metadata['source']:30s} — {len(doc.page_content):,} chars")

✅ Knowledge base built : 4 documents
   module3_press_analysis         — 1,371 chars
   module2_gnn                    — 854 chars
   module2_patterns               — 1,613 chars
   barca_philosophy               — 1,831 chars


In [ ]:
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)

for m in genai.list_models():
    if 'embed' in m.name.lower():
        print(m.name, '|', m.supported_generation_methods)

models/gemini-embedding-001 | ['embedContent', 'countTextTokens', 'countTokens', 'asyncBatchEmbedContent']
models/gemini-embedding-2-preview | ['embedContent', 'countTextTokens', 'countTokens', 'asyncBatchEmbedContent']


In [ ]:
!pip install sentence-transformers -q
print("✅ Done")

✅ Done


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

print("Loading embedding model (downloads ~90MB first time)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(chunks, embeddings)

# Save to Drive
vector_store.save_local(f"{rag_dir}/faiss_index")

print(f"✅ Vector store built : {len(chunks)} chunks embedded")
print(f"✅ Saved to Drive     : {rag_dir}/faiss_index")

Loading embedding model (downloads ~90MB first time)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Vector store built : 16 chunks embedded
✅ Saved to Drive     : /content/drive/MyDrive/BarçaIQ/rag/faiss_index


In [ ]:
!pip install langchain-groq -q
print("✅ Done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.7 MB/s eta 0:00:00
✅ Done


In [ ]:
from google.colab import userdata
from langchain_groq import ChatGroq

GROQ_API_KEY = userdata.get('GROQ_API_KEY')

llm = ChatGroq(
    model       = "llama-3.3-70b-versatile",
    temperature = 0.3,
    api_key     = GROQ_API_KEY
)

retriever = vector_store.as_retriever(
    search_type   = "similarity",
    search_kwargs = {"k": 4}
)

prompt_template = """You are BarçaIQ — an AI tactical assistant for FC Barcelona's
coaching staff under Hansi Flick. You answer questions using real StatsBomb data
from Barça's Pep era (2008-12) and MSN era (2014-17).

Rules:
- Only answer using the context provided below
- Always cite specific numbers and statistics from the data
- Frame answers as actionable recommendations for Flick's current squad
- If the context doesn't contain enough information, say so honestly
- Be concise and tactical — coaching staff don't want essays

Context from BarçaIQ knowledge base:
{context}

Question: {question}

Tactical Answer:"""

prompt = PromptTemplate(
    template        = prompt_template,
    input_variables = ["context", "question"]
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain ready — Llama 3.3 70B via Groq")
print("\nTesting first question...")
print("─" * 55)

response = rag_chain.invoke(
    "When should Flick's team trigger a press and how many players should press?"
)
print(response)

✅ RAG chain ready — Llama 3.3 70B via Groq

Testing first question...
───────────────────────────────────────────────────────
Trigger press within the first 2 actions of opponent possession. Optimal press uses 1-2 players (0.415 and 0.402 success rates, respectively), with the rest holding shape and covering lanes. This approach maximizes press success while minimizing risks.


In [ ]:
questions = [
    "Which Barça era had more dangerous passing patterns and why?",
    "What is the most effective tactical pattern for creating shots?",
    "How does the false nine role affect Barça's attacking patterns?"
]

for q in questions:
    print(f"\n {q}")
    print("─" * 55)
    print(rag_chain.invoke(q))
    print()


 Which Barça era had more dangerous passing patterns and why?
───────────────────────────────────────────────────────
Based on the data, the Pep era (2008-12) had more dangerous passing patterns, specifically the "Third-Man Combination" pattern, which occurred 6,420 times with a shot rate of 22.7%. This pattern's high frequency and danger make it a key area to focus on. 

To apply this to the current squad under Hansi Flick, I recommend emphasizing the "Third-Man Principle" and training players to create positional superiority through movement and spacing, allowing for effective third-man combinations. This could lead to increased shot creation and goal-scoring opportunities.


 What is the most effective tactical pattern for creating shots?
───────────────────────────────────────────────────────
Based on the data, I recommend prioritizing the THIRD-MAN COMBINATION pattern, which has a shot rate of 22.7% (6,420 sequences). This pattern is not only the most frequent but also the most d

In [ ]:
prompt_template = """You are BarçaIQ — an AI tactical assistant for FC Barcelona's
coaching staff under Hansi Flick. You answer questions using real StatsBomb data
from Barça's Pep era (2008-12) and MSN era (2014-17).

STRICT RULES:
- ONLY use numbers and statistics that appear explicitly in the context below
- NEVER invent, estimate, or calculate statistics not present in the context
- If the context lacks specific numbers to answer a question, say exactly:
  "The knowledge base does not contain specific data on this — here is what I know:"
  then share only what IS in the context
- Frame answers as actionable recommendations for Flick
- Be concise — coaching staff don't want essays

Context from BarçaIQ knowledge base:
{context}

Question: {question}

Tactical Answer (only use statistics explicitly stated in the context above):"""

prompt = PromptTemplate(
    template        = prompt_template,
    input_variables = ["context", "question"]
)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Retest the hallucination question
print(" How does the false nine role affect Barça's attacking patterns?")
print("─" * 55)
print(rag_chain.invoke(
    "How does the false nine role affect Barça's attacking patterns?"
))

❓ How does the false nine role affect Barça's attacking patterns?
───────────────────────────────────────────────────────
The knowledge base does not contain specific data on the false nine's impact on attacking patterns — here is what I know: 
The false nine creates space in behind for midfielders to run into by dropping deep to attract centre-backs. I recommend incorporating this tactic to enhance our attacking patterns, particularly when combined with the third-man combination, which has a 22.7% shot rate.


In [ ]:
# Save chain config for FastAPI later
chain_config = {
    "llm"            : "llama-3.3-70b-versatile",
    "embeddings"     : "sentence-transformers/all-MiniLM-L6-v2",
    "vector_store"   : "FAISS",
    "chunks"         : 16,
    "retriever_k"    : 4,
    "temperature"    : 0.3,
}

with open(f"{rag_dir}/rag_config.json", 'w') as f:
    json.dump(chain_config, f, indent=2)

print("✅ RAG config saved")
print()

# ── Final demo — 3 Flick-specific questions ───────────────────────────────────
demo_questions = [
    "How should Flick set up his press to maximise success rate?",
    "What passing pattern should Flick drill most in training?",
    "Is counterpress a reliable strategy for Barça?"
]

print("=" * 55)
print("  BARCAIQ TACTICAL ASSISTANT — LIVE DEMO")
print("=" * 55)

for q in demo_questions:
    print(f"\n❓ {q}")
    print("─" * 55)
    print(rag_chain.invoke(q))
    print()

print("─" * 55)
print("  MODULE 5 COMPLETE ✅")
print("  Next → Untitled8: FastAPI ML Service")
print("─" * 55)

✅ RAG config saved

  BARCAIQ TACTICAL ASSISTANT — LIVE DEMO

❓ How should Flick set up his press to maximise success rate?
───────────────────────────────────────────────────────
To maximize press success rate, I recommend Flick to trigger press within the first 2 actions of opponent possession, using 1-2 players to initiate the press while the rest hold shape and cover lanes. Specifically, mid-third pressing (x=40-80) is optimal, with a success rate of 33.1%. Avoid mass pressing (6+ players) as it drops to 15.5% success.


❓ What passing pattern should Flick drill most in training?
───────────────────────────────────────────────────────
Flick should prioritize drilling third-man combinations and false nine drops, as they are nearly 50% more likely to end in a shot (22.7% and 21.1%) compared to the dataset average (15.1%).


❓ Is counterpress a reliable strategy for Barça?
───────────────────────────────────────────────────────
Hansi, the knowledge base does not contain specific data 

In [ ]:
# Filter to actual pass events only
pass_events = passes[passes['type'] == 'Pass'].copy()

# ── Per player pass volume by era ──
player_passes = pass_events.groupby(['player', 'era']).agg(
    total_passes    = ('id', 'count'),
    avg_pass_length = ('pass_length', 'mean'),
    through_balls   = ('pass_through_ball', 'sum'),
    goal_assists    = ('pass_goal_assist', 'sum'),
    shot_assists    = ('pass_shot_assist', 'sum'),
    cross_count     = ('pass_cross', 'sum'),
).reset_index()

# ── Who received most passes (most involved) ──
receives = pass_events.groupby(['pass_recipient', 'era']).size().reset_index()
receives.columns = ['player', 'era', 'passes_received']

# ── Merge ──
player_stats = player_passes.merge(receives, on=['player','era'], how='left')
player_stats['involvement'] = player_stats['total_passes'] + player_stats['passes_received'].fillna(0)

# ── Top 10 per era ──
pep_top = player_stats[player_stats['era']=='pep'].nlargest(10, 'involvement')
msn_top = player_stats[player_stats['era']=='msn'].nlargest(10, 'involvement')

print("=== PEP ERA TOP 10 ===")
print(pep_top[['player','total_passes','passes_received','involvement','goal_assists','shot_assists','through_balls']].to_string())
print("\n=== MSN ERA TOP 10 ===")
print(msn_top[['player','total_passes','passes_received','involvement','goal_assists','shot_assists','through_balls']].to_string())

=== PEP ERA TOP 10 ===
                            player  total_passes  passes_received  involvement goal_assists shot_assists through_balls
79          Xavier Hernández Creus         10411            10477        20888           47          278            81
16           Daniel Alves da Silva          7915             7539        15454           38          169            35
45  Lionel Andrés Messi Cuccittini          6698             8649        15347           55          233           126
8             Andrés Iniesta Luján          6008             6842        12850           23          136            53
70        Sergio Busquets i Burgos          6646             6148        12794            6           58            16
29           Gerard Piqué Bernabéu          5046             4551         9597         True           14             4
22       Eric-Sylvain Bilal Abidal          4709             3805         8514            7           31             7
12        Carles Puyol i 

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

player_doc = """
PLAYER INVOLVEMENT ANALYSIS — BARCAIQ STATISTICAL FINDINGS

=== PEP ERA (2008-12) — TOP PLAYERS BY INVOLVEMENT ===

1. XAVI HERNÁNDEZ — 20,888 total involvements (10,411 passes + 10,477 received)
   - 47 goal assists, 278 shot assists, 81 through balls
   - The undisputed orchestrator of Pep's tiki-taka system
   - Most involved player by a massive margin — 35% more than Dani Alves in second

2. DANI ALVES — 15,454 involvements (7,915 passes + 7,539 received)
   - 38 goal assists, 169 shot assists, 35 through balls

3. LIONEL MESSI — 15,347 involvements (6,698 passes + 8,649 received)
   - 55 goal assists, 233 shot assists, 126 through balls
   - Highest through ball count in Pep era — false nine creator
   - Received more passes than he made — central to all sequences

4. ANDRÉS INIESTA — 12,850 involvements (6,008 passes + 6,842 received)
   - 23 goal assists, 136 shot assists, 53 through balls

5. SERGIO BUSQUETS — 12,794 involvements (6,646 passes + 6,148 received)
   - 6 goal assists, 58 shot assists, 16 through balls
   - Highest pass count of defensive players — pivot of Pep's system

=== MSN ERA (2014-17) — TOP PLAYERS BY INVOLVEMENT ===

1. LIONEL MESSI — 11,695 involvements (4,975 passes + 6,720 received)
   - 41 goal assists, 195 shot assists, 146 through balls
   - Highest through balls in MSN era — most creative player

2. SERGIO BUSQUETS — 11,395 involvements (6,035 passes + 5,360 received)
   - Replaced Xavi as primary distributor in MSN era

3. JORDI ALBA — 9,886 involvements (5,321 passes + 4,565 received)
   - 13 goal assists, 56 shot assists
   - Left back overlap with Messi was critical to MSN attacking play

4. NEYMAR — 9,496 involvements (3,984 passes + 5,512 received)
   - 30 goal assists, 200 shot assists, 36 through balls
   - Second highest shot assists — devastating wide threat

5. IVAN RAKITIĆ — 9,294 involvements
   - 13 goal assists, 83 shot assists

=== MOST IMPORTANT PLAYERS BY ERA ===
- Pep era: Xavi (orchestrator, 20,888 involvements), Messi (creator, 126 through balls), Iniesta (connector)
- MSN era: Busquets (pivot, replaced Xavi), Messi (creator, 146 through balls), Neymar (200 shot assists)

=== KEY COMPARISONS ===
- Xavi (Pep): 10,411 passes, 81 through balls — creative distributor
- Busquets (MSN): 6,035 passes, 14 through balls — defensive pivot
- Messi through balls: 126 (Pep) vs 146 (MSN) — more direct in MSN era
- Messi-Neymar combined 395 shot assists — most dangerous attacking pair in MSN era
"""

embeddings  = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
faiss_path  = '/content/drive/MyDrive/BarçaIQ/rag/faiss_index'
vectorstore = FAISS.load_local(faiss_path, embeddings, allow_dangerous_deserialization=True)

doc = Document(page_content=player_doc, metadata={"source": "player_involvement_analysis"})
vectorstore.add_documents([doc])
vectorstore.save_local(faiss_path)

print("✅ Player involvement document added to FAISS")
print(f"Total chunks: {vectorstore.index.ntotal}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Player involvement document added to FAISS
Total chunks: 17


In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load existing FAISS index
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
faiss_path = '/content/drive/MyDrive/BarçaIQ/rag/faiss_index'
vectorstore = FAISS.load_local(faiss_path, embeddings, allow_dangerous_deserialization=True)

# Add new document
doc = Document(
    page_content=player_doc,
    metadata={"source": "player_involvement_analysis", "type": "statistical"}
)
vectorstore.add_documents([doc])

# Save updated index
vectorstore.save_local(faiss_path)
print("✅ Player involvement document added to FAISS")
print(f"Total chunks now: {vectorstore.index.ntotal}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Player involvement document added to FAISS
Total chunks now: 18
